<a href="https://colab.research.google.com/github/TaherBenAfia/Fly2/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")  # move from notebooks/ to the repo root

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-starter
Starter data found. You're ready.


In [2]:
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [3]:
import numpy as np

numeric_features = ['search_volume','competition','cpc','word_count','char_count',
    'impressions_90d','clicks_90d','sessions_90d','ai_sessions_90d',
    'days_with_impressions','days_with_sessions','content_age_days',
    'days_since_last_update','ctr','avg_position','engagement_rate',
    'scroll_rate','ai_traffic_pct']
categorical_features = ['competition_level','content_type','main_intent','age_tier',
    'freshness_tier','word_count_tier','impression_tier','position_tier']
numeric_features, categorical_features

(['search_volume',
  'competition',
  'cpc',
  'word_count',
  'char_count',
  'impressions_90d',
  'clicks_90d',
  'sessions_90d',
  'ai_sessions_90d',
  'days_with_impressions',
  'days_with_sessions',
  'content_age_days',
  'days_since_last_update',
  'ctr',
  'avg_position',
  'engagement_rate',
  'scroll_rate',
  'ai_traffic_pct'],
 ['competition_level',
  'content_type',
  'main_intent',
  'age_tier',
  'freshness_tier',
  'word_count_tier',
  'impression_tier',
  'position_tier'])

In [4]:
df.loc[df['avg_position'] == 0, 'avg_position'] = np.nan
df['has_keyword_data'] = df['search_volume'].notna().astype(int)
X_num = df[numeric_features].fillna(0)
for col in ['impressions_90d','clicks_90d','sessions_90d','ai_sessions_90d']:
    X_num['log_' + col] = np.log1p(X_num[col])
X_num

,search_volume,competition,cpc,word_count,char_count,impressions_90d,clicks_90d,sessions_90d,ai_sessions_90d,days_with_impressions,...,days_since_last_update,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,log_impressions_90d,log_clicks_90d,log_sessions_90d,log_ai_sessions_90d
0,10.0,0.67,2.05,3221.0,20457.0,3803,29,17,0,88,...,20,0.76,10.6,5.88,4.55,0.00,8.243808,3.401197,2.890372,0.000000
1,90.0,0.01,0.05,2481.0,15562.0,15320,7,9,0,88,...,25,0.05,20.3,0.00,10.00,0.00,9.636980,2.079442,2.302585,0.000000
2,0.0,0.00,0.00,3515.0,23643.0,12581,11,11,0,88,...,20,0.09,36.5,0.00,28.57,0.00,9.440023,2.484907,2.484907,0.000000
3,10.0,0.00,0.00,0.0,0.0,11751,58,78,0,88,...,22,0.49,6.2,1.28,3.45,0.00,9.371779,4.077537,4.369448,0.000000
4,0.0,0.00,0.00,2803.0,17469.0,19140,24,145,0,88,...,14,0.13,44.0,0.00,24.29,0.00,9.859588,3.218876,4.983607,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29995,10.0,0.05,0.00,1386.0,9084.0,1,0,3,0,1,...,20,0.00,0.0,0.00,0.00,0.00,0.693147,0.000000,1.386294,0.000000
29996,0.0,0.00,0.00,2654.0,17056.0,761,3,3,0,77,...,20,0.39,6.6,0.00,66.67,0.00,6.635947,1.386294,1.386294,0.000000
29997,10.0,1.00,0.00,2857.0,18725.0,6336,12,44,0,88,...,20,0.19,4.1,0.00,0.00,0.00,8.754161,2.564949,3.806662,0.000000
29998,10.0,0.00,0.00,0.0,0.0,154763,340,463,0,88,...,22,0.22,6.0,1.73,4.06,0.00,11.949657,5.831882,6.139885,0.000000


In [5]:
X_cat = pd.get_dummies(df[categorical_features].fillna('unknown'), drop_first=False)
X = pd.concat([X_num, X_cat, df[['has_keyword_data']]], axis=1)
X.shape, X.isna().sum().sum()

((30000, 57), np.int64(0))

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [6]:
print(df.isna().sum().sort_values(ascending=False))
print(df.groupby('content_type')[['search_volume','competition','cpc']].apply(lambda g: g.isna().mean().round(3)))

provider_used             21438
word_count                 7699
char_count                 7699
char_count_tier            7699
word_count_tier            7699
model_used                 5733
trend_pct                  3388
competition_level          2610
cpc                        2468
search_volume              2468
competition                2468
main_intent                2374
avg_position               1205
scroll_rate                 125
content_type                  0
content_id                    0
client_id                     0
impressions_90d               0
clicks_90d                    0
sessions_90d                  0
pageviews_90d                 0
days_with_sessions            0
impressions_last_30d          0
clicks_last_30d               0
users_90d                     0
engaged_sessions_90d          0
ai_sessions_90d               0
scroll_events_90d             0
days_with_impressions         0
content_age_days              0
sessions_prev_30d             0
clicks_p

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [7]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

y = df['trend_direction'].str.lower().eq('down').astype(int)
candidates = numeric_features + ['trend_pct']

for col in candidates:
    x = df[[col]].fillna(0)
    t = DecisionTreeClassifier(max_depth=1, random_state=42).fit(x, y)
    auc = roc_auc_score(y, t.predict_proba(x)[:, 1])
    print(col, round(auc, 3))

search_volume 0.527
competition 0.506
cpc 0.501
word_count 0.566
char_count 0.563
impressions_90d 0.581
clicks_90d 0.54
sessions_90d 0.53
ai_sessions_90d 0.501
days_with_impressions 0.583
days_with_sessions 0.53
content_age_days 0.586
days_since_last_update 0.547
ctr 0.54
avg_position 0.545
engagement_rate 0.505
scroll_rate 0.513
ai_traffic_pct 0.502
trend_pct 1.0


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [8]:
excluded = {
    'trend_direction': 'label',
    'trend_pct': 'label source, AUC=1.0 alone',
    'content_id': 'identifier',
    'client_id': 'identifier',
    'provider_used': 'not a model feature per data dictionary',
    'model_used': 'not a model feature per data dictionary'
}
excluded

{'trend_direction': 'label',
 'trend_pct': 'label source, AUC=1.0 alone',
 'content_id': 'identifier',
 'client_id': 'identifier',
 'provider_used': 'not a model feature per data dictionary',
 'model_used': 'not a model feature per data dictionary'}

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.